# W4111 — ClassicModels API Notebook (Task 6)

This notebook exercises all implemented resources and HTTP routes.

**Before running:** 
1. Copy `.env.example` → `.env` and fill in your MySQL credentials.
2. Start the server in a separate terminal: `uvicorn app.main:app --reload`
3. Run cells top-to-bottom.


## 0 — Setup

In [1]:
import sys, os
from pathlib import Path

# Make sure the project root is on the path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env so DB_* variables are available
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

BASE_URL = "http://127.0.0.1:8000"
print("Root:", ROOT)
print("DB_HOST:", os.getenv("DB_HOST"))
print("DB_NAME:", os.getenv("DB_NAME"))

Root: /Users/tejal/Desktop/W4111-Template_Web_Application
DB_HOST: localhost
DB_NAME: classicmodels


## 1 — MySQLDataService (direct, no HTTP)

In [2]:
from app.services.MySQLDataService import MySQLDataService

svc = MySQLDataService({"table": "customers", "primary_key": "customerNumber"})

# retrieveByTemplate — all customers in USA
usa_customers = svc.retrieveByTemplate({"country": "USA"})
print(f"USA customers: {len(usa_customers)}")
for c in usa_customers[:3]:
    print(" ", c["customerNumber"], c["customerName"])

USA customers: 36
  112 Signal Gift Stores
  124 Mini Gifts Distributors Ltd.
  129 Mini Wheels Co.


In [3]:
# retrieveByPrimaryKey
row = svc.retrieveByPrimaryKey("103")
print("Customer 103:", row)

Customer 103: {'customerNumber': 103, 'customerName': 'Atelier graphique', 'contactLastName': 'Schmitt', 'contactFirstName': 'Carine ', 'phone': '40.32.2555', 'addressLine1': '54, rue Royale', 'addressLine2': None, 'city': 'Nantes', 'state': None, 'postalCode': '44000', 'country': 'France', 'salesRepEmployeeNumber': 1370, 'creditLimit': Decimal('21000.00')}


## 2 — CustomerResource (direct, no HTTP)

In [4]:
from app.resources.CustomerResource import Customer, CustomerResource

cust_resource = CustomerResource()

# get() with template
collection = cust_resource.get({"country": "France"})
print(f"French customers: {len(collection.items)}")
for c in collection.items[:3]:
    print(" ", c.customerNumber, c.customerName)

French customers: 12
  103 Atelier graphique
  119 La Rochelle Gifts
  146 Saveley & Henriot, Co.


In [5]:
# get_by_id()
c = cust_resource.get_by_id("103")
print(c)

customerNumber=103 customerName='Atelier graphique' contactLastName='Schmitt' contactFirstName='Carine ' phone='40.32.2555' addressLine1='54, rue Royale' addressLine2=None city='Nantes' state=None postalCode='44000' country='France' salesRepEmployeeNumber=1370 creditLimit=21000.0


In [6]:
# post() — create a new customer
new_cust = Customer(
    customerNumber=99999,
    customerName="Test Customer",
    contactLastName="Smith",
    contactFirstName="Alice",
    phone="555-0100",
    addressLine1="1 Test Ave",
    city="New York",
    country="USA",
    creditLimit=5000.00,
)
new_id = cust_resource.post(new_cust)
print("Created customer with key:", new_id)

Created customer with key: 99999


In [7]:
# put() — update that customer
updated_cust = new_cust.model_copy(update={"creditLimit": 9999.99})
n = cust_resource.put("99999", updated_cust)
print("Rows updated:", n)
print(cust_resource.get_by_id("99999"))

Rows updated: 1
customerNumber=99999 customerName='Test Customer' contactLastName='Smith' contactFirstName='Alice' phone='555-0100' addressLine1='1 Test Ave' addressLine2=None city='New York' state=None postalCode=None country='USA' salesRepEmployeeNumber=None creditLimit=9999.99


In [8]:
# delete()
n = cust_resource.delete("99999")
print("Rows deleted:", n)

Rows deleted: 1


## 3 — OrderResource (direct, no HTTP)

In [9]:
from app.resources.OrderResource import Order, OrderResource

order_resource = OrderResource()

# get() all Shipped orders
shipped = order_resource.get({"status": "Shipped"})
print(f"Shipped orders: {len(shipped.items)}")
for o in shipped.items[:3]:
    print(" ", o.orderNumber, o.orderDate, o.customerNumber)

Shipped orders: 303
  10100 2003-01-06 363
  10101 2003-01-09 128
  10102 2003-01-10 181


In [10]:
# get_by_id()
o = order_resource.get_by_id("10100")
print(o)

orderNumber=10100 orderDate=datetime.date(2003, 1, 6) requiredDate=datetime.date(2003, 1, 13) shippedDate=datetime.date(2003, 1, 10) status='Shipped' comments=None customerNumber=363


In [11]:
# post() — create a test order
new_order = Order(
    orderNumber=99001,
    orderDate="2025-01-01",
    requiredDate="2025-01-15",
    status="In Process",
    customerNumber=103,
)
key = order_resource.post(new_order)
print("Created order:", key)

Created order: 99001


In [12]:
# put() — update status
updated_order = new_order.model_copy(update={"status": "Shipped", "shippedDate": "2025-01-10"})
n = order_resource.put("99001", updated_order)
print("Rows updated:", n)

Rows updated: 1


In [13]:
# delete()
n = order_resource.delete("99001")
print("Rows deleted:", n)

Rows deleted: 1


## 4 — OrderDetailsResource (direct, composite PK)

In [14]:
from app.resources.OrderDetailsResource import OrderDetail, OrderDetailsResource

od_resource = OrderDetailsResource()

# get all lines for order 10100
lines = od_resource.get_by_order("10100")
print(f"Lines for order 10100: {len(lines.items)}")
for line in lines.items:
    print(" ", line.productCode, line.quantityOrdered, line.priceEach)

Lines for order 10100: 4
  S18_1749 30 136.0
  S18_2248 50 55.09
  S18_4409 22 75.46
  S24_3969 49 35.29


In [15]:
# get single line by composite PK
detail = od_resource.get_by_order_and_product("10100", "S18_1749")
print(detail)

orderNumber=10100 productCode='S18_1749' quantityOrdered=30 priceEach=136.0 orderLineNumber=3


In [16]:
# post() — need a real order first; reuse 99001 pattern with cleanup
from app.resources.OrderResource import Order, OrderResource
order_resource = OrderResource()
order_resource.post(Order(
    orderNumber=99001, orderDate="2025-01-01", requiredDate="2025-01-15",
    status="In Process", customerNumber=103
))

new_line = OrderDetail(
    orderNumber=99001, productCode="S18_1749",
    quantityOrdered=10, priceEach=150.00, orderLineNumber=1
)
key = od_resource.post(new_line)
print("Created detail:", key)

Created detail: 99001|S18_1749


In [17]:
# put() — update quantity
updated_line = new_line.model_copy(update={"quantityOrdered": 25})
n = od_resource.put("99001", "S18_1749", updated_line)
print("Rows updated:", n)
print(od_resource.get_by_order_and_product("99001", "S18_1749"))

Rows updated: 1
orderNumber=99001 productCode='S18_1749' quantityOrdered=25 priceEach=150.0 orderLineNumber=1


In [18]:
# delete() detail then order
print("Detail deleted:", od_resource.delete("99001", "S18_1749"))
print("Order deleted:", order_resource.delete("99001"))

Detail deleted: 1
Order deleted: 1


## 5 — HTTP API via httpx

Make sure `uvicorn app.main:app --reload` is running before executing these cells.

In [19]:
import httpx, json

def pretty(r):
    print(f"HTTP {r.status_code}")
    try:
        print(json.dumps(r.json(), indent=2, default=str))
    except Exception:
        print(r.text)

In [20]:
# GET /health
pretty(httpx.get(f"{BASE_URL}/health"))

HTTP 200
{
  "status": "ok"
}


In [21]:
# GET /customers — list all
r = httpx.get(f"{BASE_URL}/customers")
data = r.json()
print(f"HTTP {r.status_code} — total customers: {len(data['items'])}")
for c in data["items"][:3]:
    print(" ", c["customerNumber"], c["customerName"])

HTTP 200 — total customers: 122
  103 Atelier graphique
  112 Signal Gift Stores
  114 Australian Collectors, Co.


In [22]:
# GET /customers?country=France
r = httpx.get(f"{BASE_URL}/customers", params={"country": "France"})
data = r.json()
print(f"French customers: {len(data['items'])}")

French customers: 12


In [23]:
# GET /customers/103
pretty(httpx.get(f"{BASE_URL}/customers/103"))

HTTP 200
{
  "customerNumber": 103,
  "customerName": "Atelier graphique",
  "contactLastName": "Schmitt",
  "contactFirstName": "Carine ",
  "phone": "40.32.2555",
  "addressLine1": "54, rue Royale",
  "addressLine2": null,
  "city": "Nantes",
  "state": null,
  "postalCode": "44000",
  "country": "France",
  "salesRepEmployeeNumber": 1370,
  "creditLimit": 21000.0
}


In [24]:
# POST /customers — create
payload = {
    "customerNumber": 99999,
    "customerName": "HTTP Test Customer",
    "contactLastName": "Jones",
    "contactFirstName": "Bob",
    "phone": "555-9999",
    "addressLine1": "99 API Lane",
    "city": "New York",
    "country": "USA",
    "creditLimit": 1000.00,
}
pretty(httpx.post(f"{BASE_URL}/customers", json=payload))

HTTP 200
"99999"


In [25]:
# PUT /customers/99999
payload["creditLimit"] = 7500.00
pretty(httpx.put(f"{BASE_URL}/customers/99999", json=payload))

HTTP 200
{
  "updated": 1
}


In [26]:
# GET /customers/99999 — verify update
pretty(httpx.get(f"{BASE_URL}/customers/99999"))

HTTP 200
{
  "customerNumber": 99999,
  "customerName": "HTTP Test Customer",
  "contactLastName": "Jones",
  "contactFirstName": "Bob",
  "phone": "555-9999",
  "addressLine1": "99 API Lane",
  "addressLine2": null,
  "city": "New York",
  "state": null,
  "postalCode": null,
  "country": "USA",
  "salesRepEmployeeNumber": null,
  "creditLimit": 7500.0
}


In [27]:
# DELETE /customers/99999
pretty(httpx.delete(f"{BASE_URL}/customers/99999"))

HTTP 200
{
  "deleted": 1
}


In [28]:
# GET /customers/99999 — expect 404
r = httpx.get(f"{BASE_URL}/customers/99999")
print(f"HTTP {r.status_code} (expected 404)")

HTTP 404 (expected 404)


In [29]:
# GET /orders — all orders
r = httpx.get(f"{BASE_URL}/orders")
data = r.json()
print(f"Total orders: {len(data['items'])}")
for o in data["items"][:3]:
    print(" ", o["orderNumber"], o["status"], o["customerNumber"])

Total orders: 326
  10100 Shipped 363
  10101 Shipped 128
  10102 Shipped 181


In [30]:
# GET /orders?status=Shipped
r = httpx.get(f"{BASE_URL}/orders", params={"status": "Shipped"})
print(f"Shipped orders: {len(r.json()['items'])}")

Shipped orders: 303


In [31]:
# GET /orders/10100
pretty(httpx.get(f"{BASE_URL}/orders/10100"))

HTTP 200
{
  "orderNumber": 10100,
  "orderDate": "2003-01-06",
  "requiredDate": "2003-01-13",
  "shippedDate": "2003-01-10",
  "status": "Shipped",
  "comments": null,
  "customerNumber": 363
}


In [32]:
# POST /orders — create test order
order_payload = {
    "orderNumber": 99001,
    "orderDate": "2025-01-01",
    "requiredDate": "2025-01-15",
    "status": "In Process",
    "customerNumber": 103,
}
pretty(httpx.post(f"{BASE_URL}/orders", json=order_payload))

HTTP 200
"99001"


In [33]:
# GET /orders/99001/orderdetails — empty (no lines yet)
pretty(httpx.get(f"{BASE_URL}/orders/99001/orderdetails"))

HTTP 200
{
  "items": []
}


In [34]:
# POST /orderdetails — add a line
detail_payload = {
    "orderNumber": 99001,
    "productCode": "S18_1749",
    "quantityOrdered": 10,
    "priceEach": 150.00,
    "orderLineNumber": 1,
}
pretty(httpx.post(f"{BASE_URL}/orderdetails", json=detail_payload))

HTTP 200
"99001|S18_1749"


In [35]:
# GET /orders/99001/orderdetails/S18_1749
pretty(httpx.get(f"{BASE_URL}/orders/99001/orderdetails/S18_1749"))

HTTP 200
{
  "orderNumber": 99001,
  "productCode": "S18_1749",
  "quantityOrdered": 10,
  "priceEach": 150.0,
  "orderLineNumber": 1
}


In [36]:
# PUT /orders/99001/orderdetails/S18_1749 — update quantity
detail_payload["quantityOrdered"] = 25
pretty(httpx.put(f"{BASE_URL}/orders/99001/orderdetails/S18_1749", json=detail_payload))

HTTP 200
{
  "updated": 1
}


In [37]:
# DELETE /orders/99001/orderdetails/S18_1749
pretty(httpx.delete(f"{BASE_URL}/orders/99001/orderdetails/S18_1749"))

HTTP 200
{
  "deleted": 1
}


In [38]:
# PUT /orders/99001 — update status
order_payload["status"] = "Cancelled"
pretty(httpx.put(f"{BASE_URL}/orders/99001", json=order_payload))

HTTP 200
{
  "updated": 1
}


In [39]:
# DELETE /orders/99001 — cleanup
pretty(httpx.delete(f"{BASE_URL}/orders/99001"))

HTTP 200
{
  "deleted": 1
}


In [40]:
# 404 test — order that doesn't exist
r = httpx.get(f"{BASE_URL}/orders/99001")
print(f"HTTP {r.status_code} (expected 404): {r.json()['detail']}")

HTTP 404 (expected 404): No order with orderNumber '99001'
